In [ ]:
!pip install google-play-scraper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.9 MB/s eta 0:00:00


In [ ]:
from google_play_scraper import app, Sort, reviews_all
import pandas as pd
import numpy as np
import json, os, uuid


In [ ]:
from google_play_scraper import reviews, Sort
import pandas as pd

all_2025 = []
continuation_token = None

while True:
    result, continuation_token = reviews(
        'com.google.android.youtube',
        lang='en',
        country='us',
        sort=Sort.NEWEST,
        count=200,
        continuation_token=continuation_token
    )

    for r in result:
        review_date = pd.to_datetime(r['at'])

        if review_date.year < 2025:
            print("Semua review tahun 2025 telah diambil.")
            break

        if review_date.year == 2025:
            all_2025.append(r)

    if review_date.year < 2025 or continuation_token is None:
        break

print(f"Total review 2025 terkumpul: {len(all_2025)}")

Semua review tahun 2025 telah diambil.
Total review 2025 terkumpul: 1058464


In [ ]:
# Ubah list of dict menjadi DataFrame
df = pd.DataFrame(all_2025)

# Tambahkan kolom tanggal, bulan, tahun
df['review_date'] = pd.to_datetime(df['at'])
df['month'] = df['review_date'].dt.month
df['year'] = df['review_date'].dt.year

# Rename kolom sesuai permintaan
df = df.rename(columns={
    'score': 'rating',
    'userName': 'user_name',
    'reviewId': 'review_id',
    'content': 'review_description',
    'at': 'review_date',
    'replyContent': 'developer_response',
    'repliedAt': 'developer_response_date',
    'thumbsUpCount': 'thumbs_up'
})

# Tambahkan kolom tambahan
df['source'] = 'Google Play'

In [ ]:
df

,review_id,user_name,userImage,review_description,rating,thumbs_up,reviewCreatedVersion,review_date,developer_response,developer_response_date,appVersion,review_date,month,year,source
0,f61485f4-e3e3-409b-b152-fd95f6039289,Kimberlie Pagán,https://play-lh.googleusercontent.com/a-/ALV-U...,ever since yall took away the block button for...,1,1,20.51.39,2025-12-31 23:59:14,None,None,20.51.39,2025-12-31 23:59:14,12,2025,Google Play
1,cfb78be2-e5ab-4b68-a127-a24333626e5a,MD Mahide Hassan,https://play-lh.googleusercontent.com/a/ACg8oc...,Nice,5,0,20.49.44,2025-12-31 23:58:33,None,None,20.49.44,2025-12-31 23:58:33,12,2025,Google Play
2,c8a98639-231b-4f62-9cb2-eca28a09c57c,Sunday Nwolisa,https://play-lh.googleusercontent.com/a/ACg8oc...,A nice app,5,0,10.18.55,2025-12-31 23:54:39,None,None,10.18.55,2025-12-31 23:54:39,12,2025,Google Play
3,4f2a89d3-a1ca-4bdc-8c94-2bb7a77d8ac0,Ramki Ramki,https://play-lh.googleusercontent.com/a/ACg8oc...,ஃபைவ்,1,0,20.51.39,2025-12-31 23:52:30,None,None,20.51.39,2025-12-31 23:52:30,12,2025,Google Play
4,fea42746-01d2-4a81-97cc-9d62f32e45a5,Steve Seetahal,https://play-lh.googleusercontent.com/a-/ALV-U...,"2 minor bugs observed , with this update 1 bac...",2,0,20.51.39,2025-12-31 23:50:17,None,None,20.51.39,2025-12-31 23:50:17,12,2025,Google Play
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1058459,7e400d15-3301-420c-9295-2fab9bf410a8,Drazen Bjelovuk,https://play-lh.googleusercontent.com/a-/ALV-U...,"As with most Google apps, settings/options are...",1,7,19.50.40,2025-01-01 00:08:12,None,None,19.50.40,2025-01-01 00:08:12,1,2025,Google Play
1058460,2bcd54d3-1f89-4dac-857b-476be2dcefec,Agbetrobou Oliviare,https://play-lh.googleusercontent.com/a/ACg8oc...,YouTube Oliver,5,0,10.49.59,2025-01-01 00:07:50,None,None,10.49.59,2025-01-01 00:07:50,1,2025,Google Play
1058461,08210768-e806-4071-8d9a-1106c08f76c2,Chibuike Uche,https://play-lh.googleusercontent.com/a/ACg8oc...,YoTube,5,0,13.07.55,2025-01-01 00:05:53,None,None,13.07.55,2025-01-01 00:05:53,1,2025,Google Play
1058462,19148b96-24f9-4cb3-b43d-79b2b5030227,Steven E,https://play-lh.googleusercontent.com/a/ACg8oc...,Bloatware,1,0,None,2025-01-01 00:05:18,None,None,None,2025-01-01 00:05:18,1,2025,Google Play


In [ ]:
df.to_csv('youtube_reviews_2025_all.csv', index=False)
print("CSV berhasil disimpan.")

CSV berhasil disimpan.


In [ ]:
# =========================
# 3️⃣ Filter review terlalu pendek (misal <20 karakter)
# =========================
MIN_REVIEW_LENGTH = 20
df_minLength = df[df['review_description'].str.len() >= MIN_REVIEW_LENGTH]

# =========================
# 4️⃣ Sampling 400 review per bulan
# =========================
SAMPLE_PER_MONTH = 500

df_sample = (
    df_minLength.groupby('month', group_keys=False)
      .apply(lambda x: x.sample(
          n=min(SAMPLE_PER_MONTH, len(x)),
          random_state=42
      ))
)

df_sample.reset_index(drop=True, inplace=True)

# Cek distribusi per bulan
print(df_sample['month'].value_counts().sort_index())

month
1     500
2     500
3     500
4     500
5     500
6     500
7     500
8     500
9     500
10    500
11    500
12    500
Name: count, dtype: int64


/tmp/ipython-input-3613488669.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


In [ ]:
df_sample

,review_id,user_name,userImage,review_description,rating,thumbs_up,reviewCreatedVersion,review_date,developer_response,developer_response_date,appVersion,review_date,month,year,source
0,eeb4ad6a-39d9-4576-98b6-6ee2d769230d,Seth (Omashog),https://play-lh.googleusercontent.com/a-/ALV-U...,"Premium user here, and my best advice is to go...",1,190,19.50.40,2025-01-10 00:51:21,None,None,19.50.40,2025-01-10 00:51:21,1,2025,Google Play
1,49b378c1-d103-40c7-bf16-a6645cebb291,Abc,https://play-lh.googleusercontent.com/a-/ALV-U...,how to volume problem,5,0,19.50.40,2025-01-17 15:49:56,None,None,19.50.40,2025-01-17 15:49:56,1,2025,Google Play
2,7e21598c-904a-454b-a0c2-7fc286d29869,Anwsr Anear,https://play-lh.googleusercontent.com/a/ACg8oc...,YouTube se pranam MDANOWAR,5,0,17.11.34,2025-01-02 02:29:14,None,None,17.11.34,2025-01-02 02:29:14,1,2025,Google Play
3,1ee11782-e06d-4183-b970-71e65b4e198f,Firoz Firoz,https://play-lh.googleusercontent.com/a/ACg8oc...,Naam firoj Naam Pramod Firoj Nagpur,3,1,19.47.51,2025-01-21 22:56:47,None,None,19.47.51,2025-01-21 22:56:47,1,2025,Google Play
4,b795a6f0-be74-4af5-8469-8ee2a1013c1c,Makshood Alam,https://play-lh.googleusercontent.com/a/ACg8oc...,Please update my YouTube,1,0,10.37.58,2025-01-07 09:41:30,None,None,10.37.58,2025-01-07 09:41:30,1,2025,Google Play
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,2d71c8e5-d955-41a1-afb6-2a95582150b5,Ajay Sharma,https://play-lh.googleusercontent.com/a/ACg8oc...,good app on the my life,3,1,20.51.39,2025-12-24 01:37:40,None,None,20.51.39,2025-12-24 01:37:40,12,2025,Google Play
5996,432c4f6f-471a-4a7a-b7b4-7f649889da34,Rowena,https://play-lh.googleusercontent.com/a/ACg8oc...,i cant open this app,1,0,12.47.58,2025-12-15 10:58:30,None,None,12.47.58,2025-12-15 10:58:30,12,2025,Google Play
5997,49eca3c0-0197-41b1-90a8-ac98f68747b6,Prakhar Gupta,https://play-lh.googleusercontent.com/a-/ALV-U...,within just 10 min I am getting 2 ads,1,0,20.51.34,2025-12-20 14:30:25,None,None,20.51.34,2025-12-20 14:30:25,12,2025,Google Play
5998,177d5392-97f7-43c4-9909-6f1f4ebfe6f2,Idowu oluwaseun,https://play-lh.googleusercontent.com/a/ACg8oc...,It was good YouTube but i don't understand why...,5,0,12.37.59,2025-12-27 09:10:28,None,None,12.37.59,2025-12-27 09:10:28,12,2025,Google Play


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df_sample.to_csv('youtube_reviews_2025_6Thousand_sample.csv', index=False)
print("CSV berhasil disimpan.")

CSV berhasil disimpan.


In [ ]:
g_df2.drop(columns=['userImage', 'reviewCreatedVersion'], inplace=True)

g_df2.rename(
    columns={
        'score': 'rating',
        'userName': 'user_name',
        'reviewId': 'review_id',
        'content': 'review_description',
        'at': 'review_date',
        'replyContent': 'developer_response',
        'repliedAt': 'developer_response_date',
        'thumbsUpCount': 'thumbs_up'
    },
    inplace=True
)

g_df2.insert(0, 'source', 'Google Play')
g_df2.insert(3, 'review_title', None)
g_df2['language_code'] = 'en'
g_df2['country_code'] = 'us'

In [ ]:
g_df2['review_date'] = pd.to_datetime(g_df2['review_date'])
g_df2.reset_index(drop=True, inplace=True)

result = g_df2
result.head()

In [ ]:
result.to_csv(
    'youtube_google_play_reviews_2024_2025.csv',
    index=False,
    encoding='utf-8-sig'
)